In [2]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from ipywidgets import interact, FloatSlider, IntSlider, Layout

## Model 1 with reproduction
def PEATONE(Y,t, kCO2, kMETHANE):
    acro = Y[0]; cato = Y[1]; co2 = Y[2]; ch4 = Y[3]
    dacro_dt=-acro*kCO2 #Acrotelm sugar consumption
    dcato_dt=-cato*kMETHANE #Catotelm sugar consumption
    dch4_dt=cato*kMETHANE #CH4 Production (inverse of catotelm sugar consumption)
    dco2_dt=acro*kCO2 #CO2 Production (inverse of acrotelm sugar consumption)
    return [dacro_dt, dcato_dt, dco2_dt, dch4_dt]

def plot_peat_model(kCO2, kMETHANE, waterlevel1, waterlevel2, changetime, endtime):
    # Constants and Initial Setup
    peat = 100
    t1 = np.linspace(0, changetime, 1000)
    t2 = np.linspace(changetime, endtime, 1000)
    
    # Initial proportions based on water level
    cato10 = peat * (waterlevel1 / 100)
    acro10 = peat - cato10
    Y10 = [acro10, cato10, 0, 0]

    # Solve First Phase
    Y1 = odeint(PEATONE, Y10, t1, args=(kCO2, kMETHANE))
    
    # Calculate Change and transition initial conditions
    change = waterlevel2 - waterlevel1
    if change >= 0:
        changesugar0 = Y1[-1, 0] * (abs(change) / 100)
        Y20 = [Y1[-1, 0] - changesugar0, Y1[-1, 1] + changesugar0, Y1[-1, 2], Y1[-1, 3]]
    else:
        changesugar0 = Y1[-1, 1] * (abs(change) / 100)
        Y20 = [Y1[-1, 0] + changesugar0, Y1[-1, 1] - changesugar0, Y1[-1, 2], Y1[-1, 3]]

    # Solve Second Phase
    Y2 = odeint(PEATONE, Y20, t2, args=(kCO2, kMETHANE))

    # Combine Results
    t_full = np.concatenate([t1, t2])
    acro_full = np.concatenate([Y1[:, 0], Y2[:, 0]])
    cato_full = np.concatenate([Y1[:, 1], Y2[:, 1]])
    co2_full = np.concatenate([Y1[:, 2], Y2[:, 2]])
    ch4_full = np.concatenate([Y1[:, 3], Y2[:, 3]])

    # Plotting
    plt.figure(figsize=(12, 6))
    plt.plot(t_full, acro_full, label='Acrotelm Sugar', color='green')
    plt.plot(t_full, cato_full, label='Catotelm Sugar', color='brown')
    plt.plot(t_full, co2_full, label='CO2 Produced', color='red', linestyle='--')
    plt.plot(t_full, ch4_full, label='CH4 Produced', color='blue', linestyle='--')
    
    plt.axvline(x=changetime, color='black', linestyle=':', label='Water Level Change')
    plt.title(f'Peatland Carbon Dynamics: WL {waterlevel1}% -> {waterlevel2}%')
    plt.xlabel('Time')
    plt.ylabel('Concentration / Mass')
    plt.legend(loc='upper right')
    plt.grid(True, alpha=0.3)
    plt.show()

# Create interactive sliders
interact(plot_peat_model,
    kCO2=FloatSlider(value=0.001, min=0.0001, max=0.001, step=0.0001, readout_format='.4f'),
    kMETHANE=FloatSlider(value=0.0005, min=0.0001, max=0.001, step=0.0001, readout_format='.4f'),
    waterlevel1=IntSlider(value=20, min=0, max=100),
    waterlevel2=IntSlider(value=50, min=0, max=100),
    changetime=IntSlider(value=1000, min=100, max=5000),
    endtime=IntSlider(value=3000, min=1000, max=10000)
);

interactive(children=(FloatSlider(value=0.001, description='kCO2', max=0.001, min=0.0001, readout_format='.4f'…